In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/colab/placement_predict_50k (5).csv')
df.head()

,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly
0,1,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.39,6.84,...,1,0,62.3,6.57,40.6,65.7,No,Medium,0,0
1,2,Male,Chennai,Tier2,ECE,AI,Yes,No,5.95,6.74,...,2,0,44.0,5.86,40.3,51.8,No,Medium,0,0
2,3,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.13,8.11,...,2,1,73.8,7.50,73.6,67.9,No,High,1,0
3,4,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.37,9.97,...,6,2,100.0,9.41,98.7,NaN,No,Excellent,1,0
4,5,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,8.25,8.99,...,4,2,90.8,9.24,83.1,100.0,No,Excellent,1,0


In [6]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "/content/drive/MyDrive/colab/placement_predict_50k (5).csv"

# --------------------------------------------------------------------------
# 1. Load + preprocess
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Remove anomaly marker if present
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

# Encode categorical columns
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))

# Fill missing numerical values
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Scale numerical features
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# --------------------------------------------------------------------------
# 2. Train / validation / test split
# --------------------------------------------------------------------------
VAL_SIZE = 0.15
TEST_SIZE = 0.15

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# --------------------------------------------------------------------------
# 3. Model benchmark
# --------------------------------------------------------------------------
def model_benchmark(X_train, y_train, X_val, y_val):
    results = []

    # ---- Model 1: Gradient Boosting ---------------------------------------
    gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )

    t0 = time.time()
    gb.fit(X_train, y_train)
    gb_fit_time = time.time() - t0

    gb_val_pred = gb.predict(X_val)
    gb_val_proba = gb.predict_proba(X_val)[:, 1]

    results.append({
        "model": "Gradient Boosting",
        "val_accuracy": accuracy_score(y_val, gb_val_pred),
        "val_f1": f1_score(y_val, gb_val_pred),
        "val_roc_auc": roc_auc_score(y_val, gb_val_proba),
        "n_estimators": gb.n_estimators,
        "fit_time_sec": round(gb_fit_time, 2)
    })

    # ---- Model 2: Random Forest -------------------------------------------
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    t0 = time.time()
    rf.fit(X_train, y_train)
    rf_fit_time = time.time() - t0

    rf_val_pred = rf.predict(X_val)
    rf_val_proba = rf.predict_proba(X_val)[:, 1]

    results.append({
        "model": "Random Forest",
        "val_accuracy": accuracy_score(y_val, rf_val_pred),
        "val_f1": f1_score(y_val, rf_val_pred),
        "val_roc_auc": roc_auc_score(y_val, rf_val_proba),
        "n_estimators": rf.n_estimators,
        "fit_time_sec": round(rf_fit_time, 2)
    })

    return pd.DataFrame(results).sort_values(
        "val_accuracy",
        ascending=False
    ).reset_index(drop=True)

# --------------------------------------------------------------------------
# 4. Run benchmark
# --------------------------------------------------------------------------
leaderboard = model_benchmark(
    X_train, y_train,
    X_val, y_val
)

print("\nValidation leaderboard (sorted by val_accuracy):")
print(leaderboard.to_string(index=False))

leaderboard.to_csv("week8_model_benchmark_results.csv", index=False)

print("\nSaved results to week8_model_benchmark_results.csv")

Train: (34999, 29) | Val: (7501, 29) | Test: (7500, 29)

Validation leaderboard (sorted by val_accuracy):
            model  val_accuracy   val_f1  val_roc_auc  n_estimators  fit_time_sec
    Random Forest      0.797227 0.784041     0.882366           200         11.44
Gradient Boosting      0.795227 0.782189     0.882184           200         39.67

Saved results to week8_model_benchmark_results.csv


In [7]:
# --------------------------------------------------------------------------
# 5. Final evaluation on the test set
# --------------------------------------------------------------------------

final_models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

print("\nFinal Test Results:")

for name, model in final_models.items():
    model.fit(X_train_val, y_train_val)

    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
    print("F1 Score:", round(f1_score(y_test, test_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

print("\nWeek 8 experiment completed successfully.")



Final Test Results:

Gradient Boosting
Accuracy: 0.7931
F1 Score: 0.7815
ROC-AUC: 0.879

Random Forest
Accuracy: 0.7929
F1 Score: 0.7804
ROC-AUC: 0.8786

Week 8 experiment completed successfully.
